<a href="https://colab.research.google.com/github/Dillybabu03/Brand-Reputation-Analysis-Indian-Brands/blob/main/DataMining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import os
from google.colab import drive

In [6]:
drive.mount('/content/drive')

PROJECT_PATH ='/content/drive/MyDrive/SMRI_project'
RAW_DATA_PATH = os.path.join(PROJECT_PATH, 'data/raw')

os.makedirs(RAW_DATA_PATH, exist_ok=True)

# Add list of brands Im going to monitor, [Zomato, Ola electric, HDFC Bank, Mama Earth, Byju's]

BRANDS =  ["Zomato", "Ola Electric", "HDFC Bank", "MamaEarth", "Byju's"]

print('Sucessfull')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Sucessfull


In [10]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

def harvest_google_news_2026(brand_name):
    print(f"Gathering 2026 news articles for: {brand_name}...")

    if brand_name == "Mamaearth":
        query_string = "(Mamaearth OR Honasa)"
    else:
        query_string = brand_name

    # We alter the web search string to look EXACTLY between Jan 1, 2026, and May 31, 2026
    encoded_query = brand_name.replace(' ', '%20')
    url = f"https://news.google.com/rss/search?q={encoded_query}%20after:2026-01-01%20before:2026-05-31&hl=en-IN&gl=IN&ceid=IN:en"

    try:
        response = requests.get(url, timeout=15)
        # BeautifulSoup acts like a digital pair of tweezers, plucking text out of raw web links
        soup = BeautifulSoup(response.content, features="xml")
        items = soup.find_all('item')

        news_records = []
        for item in items:
            news_records.append({
                'brand': brand_name,
                'source': 'Google News',
                'timestamp': item.pubDate.text,
                'title': item.title.text,
                'text': item.description.text if item.description else '',
                'engagement_score': 1
            })
        return pd.DataFrame(news_records)
    except Exception as e:
        print(f"Error fetching data for {brand_name}: {e}")
        return pd.DataFrame()

# Let's run a test loop for your 5 brands!
BRANDS = ["Zomato", "Ola Electric", "HDFC Bank", "Mamaearth", "Byju's"]

for brand in BRANDS:
    df_news = harvest_google_news_2026(brand)
    if not df_news.empty:
        # This automatically saves a spreadsheet for each brand directly into your Google Drive!
        output_file = f"/content/drive/MyDrive/SMRI_project/data/raw/{brand.lower().replace(' ', '_')}_raw_news.csv"
        df_news.to_csv(output_file, index=False)
        print(f" Successfully saved {len(df_news)} articles for {brand}!")

Gathering 2026 news articles for: Zomato...
 Successfully saved 100 articles for Zomato!
Gathering 2026 news articles for: Ola Electric...
 Successfully saved 100 articles for Ola Electric!
Gathering 2026 news articles for: HDFC Bank...
 Successfully saved 100 articles for HDFC Bank!
Gathering 2026 news articles for: Mamaearth...
 Successfully saved 91 articles for Mamaearth!
Gathering 2026 news articles for: Byju's...
 Successfully saved 100 articles for Byju's!


In [19]:
# 1. Install the specialized news reader tool
!pip install newspaper3k
!pip install lxml[html-clean]

In [23]:
import pandas as pd
from newspaper import Article
import os
import time

# Define paths matching your project directory architecture
RAW_DATA_PATH = '/content/drive/MyDrive/SMRI_project/data/raw'
BRANDS = ["zomato", "ola_electric", "hdfc_bank", "mamaearth", "byju's"]

print("Starting full-text extraction pipeline...")

# Loop through each of your 5 saved raw CSV files
for brand in BRANDS:
    file_path = os.path.join(RAW_DATA_PATH, f"{brand}_raw_news.csv")

    if os.path.exists(file_path):
        print(f"\n--- Opening {brand}_raw_news.csv to extract full text ---")
        df = pd.read_csv(file_path)

        # Create a new empty list to hold the full story text
        full_texts = []

        # Read each link inside the spreadsheet one-by-one
        for index, row in df.iterrows():
            # ==================== FIXED LINE BELOW ====================
            # We changed row['post_id'] to row['text'] since the URLs live there!
            url = row['text']
            # ==========================================================

            try:
                # Initialize the article reader tool on the URL
                article = Article(url)
                article.download()
                article.parse()

                # If text content is found, save it
                if article.text:
                    full_texts.append(article.text)
                else:
                    full_texts.append("No text content found")
            except Exception as e:
                # Handle inaccessible or blocked websites smoothly
                full_texts.append("Link inaccessible or blocked")

            # 1-second safety delay to respect host servers
            time.sleep(1)

        # Overwrite the 'text' column (which used to just be the link) with the real, full article text!
        df['text'] = full_texts

        # Overwrite the old file with our updated, rich dataset
        df.to_csv(file_path, index=False)
        print(f"Finished! Full text successfully merged into {brand}_raw_news.csv")
    else:
        print(f"Warning: File not found at {file_path}. Check your folder location.")

Starting full-text extraction pipeline...

--- Opening zomato_raw_news.csv to extract full text ---
Finished! Full text successfully merged into zomato_raw_news.csv

--- Opening ola_electric_raw_news.csv to extract full text ---
Finished! Full text successfully merged into ola_electric_raw_news.csv

--- Opening hdfc_bank_raw_news.csv to extract full text ---
Finished! Full text successfully merged into hdfc_bank_raw_news.csv

--- Opening mamaearth_raw_news.csv to extract full text ---
Finished! Full text successfully merged into mamaearth_raw_news.csv

--- Opening byju's_raw_news.csv to extract full text ---
Finished! Full text successfully merged into byju's_raw_news.csv


In [24]:


RAW_DATA_PATH = '/content/drive/MyDrive/SMRI_project/data/raw'
BRANDS = ["zomato", "ola_electric", "hdfc_bank", "mamaearth", "byju's"]

print("Fixing data columns...")

for brand in BRANDS:
    file_path = os.path.join(RAW_DATA_PATH, f"{brand}_raw_news.csv")

    if os.path.exists(file_path):
        df = pd.read_csv(file_path)

        # This instantly replaces 'Link inaccessible or blocked' with your clean Title text
        df['text'] = df['title']

        # Save the polished file back over the old one
        df.to_csv(file_path, index=False)
        print(f"✅ Fixed! {brand}_raw_news.csv now has {len(df)} rows of clean text data.")
    else:
        print(f"Could not find file for {brand}")

Fixing data columns...
✅ Fixed! zomato_raw_news.csv now has 100 rows of clean text data.
✅ Fixed! ola_electric_raw_news.csv now has 100 rows of clean text data.
✅ Fixed! hdfc_bank_raw_news.csv now has 100 rows of clean text data.
✅ Fixed! mamaearth_raw_news.csv now has 91 rows of clean text data.
✅ Fixed! byju's_raw_news.csv now has 100 rows of clean text data.
